In [47]:
import pandas as pd
import numpy as np
df = pd.read_csv("C:\\Users\\kousi\\Documents\\sentiment_analysis_fastapi_aws_docker\\dataset\\IMDB Dataset.csv")

In [48]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [49]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [50]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)


In [51]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered)

In [52]:
chat_words  = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired"
}

def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)


In [53]:
from autocorrect import Speller

spell = Speller(lang='en')

def correct_spelling_autocorrect(text):
    return spell(text)

In [54]:
correct_spelling_autocorrect("wong")

'wong'

In [55]:
import emoji

def replace_emojis(text):
    return emoji.demojize(text)

In [63]:
import re

def clean_text(text):
    """
    Cleans text for sentiment analysis:
    - Removes decorative symbols: ----, (), [], {}, /
    - Keeps sentiment-carrying punctuation: !, ?, ...
    - Replaces multiple spaces with single space
    """
    # Remove brackets but keep content inside
    text = re.sub(r'\((.*?)\)', r'\1', text)
    text = re.sub(r'\[(.*?)\]', r'\1', text)
    text = re.sub(r'\{(.*?)\}', r'\1', text)

    # Remove slashes, dashes (sequences of - or /)
    text = re.sub(r'[-/]+', ' ', text)

    # Remove other non-alphanumeric characters except ! ? . , '
    # You can adjust to keep ',' if needed
    text = re.sub(r'[^a-zA-Z0-9\s!?.]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Example
# sample = "I love this movie!!! (except the ending) ---- so sad / disappointed"
# cleaned = clean_text(sample)
# print(cleaned)

In [64]:
def preprocess(df):
    print("Started with Removing HTML Tags....")
    df["review"] = df["review"].apply(remove_html_tags)
    print("completed with Removing HTML Tags....")
    print("Started with Remove URL ....")
    df["review"] = df["review"].apply(remove_url)
    print("Completed with Remove URL ....")
    print("Started with Chat Conversion ....")
    df["review"] = df["review"].apply(chat_conversion)
    print("Completed with Chat Conversion ....")    
    print("Started with Emoji conversion ....")
    df["review"] = df["review"].apply(replace_emojis)
    print("Completed with Emoji conversion ....")
    print("Started with Cleaning Text ....")
    df["review"] = df["review"].apply(clean_text)
    print("Completed with Cleaning Text ....")    
    return df

In [65]:
df = preprocess(df)

Started with Removing HTML Tags....
completed with Removing HTML Tags....
Started with Remove URL ....
Completed with Remove URL ....
Started with Chat Conversion ....
Completed with Chat Conversion ....
Started with Emoji conversion ....
Completed with Emoji conversion ....
Started with Cleaning Text ....
Completed with Cleaning Text ....


In [66]:
import pandas as pd

sample_text = """
<p>OMG this movie was sooo good!!! 😂😂</p>
I can't------------ believe it lol. []/:::
Check this........... review here: https://www.imdb.com/title/tt0111161/
BRB watching it again. IMO one of the best movies ever!)()
"""

df_1 = pd.DataFrame({
    "review": [sample_text]
})

df_1 = preprocess(df_1)

print(df_1["review"][0])

Started with Removing HTML Tags....
completed with Removing HTML Tags....
Started with Remove URL ....
Completed with Remove URL ....
Started with Chat Conversion ....
Completed with Chat Conversion ....
Started with Emoji conversion ....
Completed with Emoji conversion ....
Started with Cleaning Text ....
Completed with Cleaning Text ....
OMG this movie was sooo good!!! facewithtearsofjoyfacewithtearsofjoy I cant believe it lol. Check this........... review here Be Right Back watching it again. In My Opinion one of the best movies ever!


In [67]:
df["review"]

0        One of the other reviewers has mentioned that ...
1        A wonderful little production. The filming tec...
2        I thought this was a wonderful way to spend Te...
3        Basically theres a family where a little boy J...
4        Petter Matteis Love in the Tears In My Eyes of...
                               ...                        
49995    I thought this movie did a down right good job...
49996    Bad plot bad dialogue bad acting idiotic direc...
49997    I am a Catholic taught in parochial elementary...
49998    Im going to have to disagree with the previous...
49999    No one expects the Star Trek movies to be high...
Name: review, Length: 50000, dtype: object

In [60]:
df["review"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wo

In [80]:
from sklearn.model_selection import train_test_split

X = df['review']
Y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [81]:
X

0        One of the other reviewers has mentioned that ...
1        A wonderful little production. The filming tec...
2        I thought this was a wonderful way to spend Te...
3        Basically theres a family where a little boy J...
4        Petter Matteis Love in the Tears In My Eyes of...
                               ...                        
49995    I thought this movie did a down right good job...
49996    Bad plot bad dialogue bad acting idiotic direc...
49997    I am a Catholic taught in parochial elementary...
49998    Im going to have to disagree with the previous...
49999    No one expects the Star Trek movies to be high...
Name: review, Length: 50000, dtype: object

In [82]:
Y

0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
49995    positive
49996    negative
49997    negative
49998    negative
49999    negative
Name: sentiment, Length: 50000, dtype: object

In [83]:
Y.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [84]:
from sklearn.preprocessing import LabelEncoder

# Example sentiment labels
y = ["positive", "negative"]

# Initialize encoder
le = LabelEncoder()

# Fit and transform labels
y_encoded = le.fit_transform(Y)

print("Original labels:", Y)
print("Encoded labels:", y_encoded)

Original labels: 0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
49995    positive
49996    negative
49997    negative
49998    negative
49999    negative
Name: sentiment, Length: 50000, dtype: object
Encoded labels: [1 1 1 ... 0 0 0]


In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [87]:
X

0        One of the other reviewers has mentioned that ...
1        A wonderful little production. The filming tec...
2        I thought this was a wonderful way to spend Te...
3        Basically theres a family where a little boy J...
4        Petter Matteis Love in the Tears In My Eyes of...
                               ...                        
49995    I thought this movie did a down right good job...
49996    Bad plot bad dialogue bad acting idiotic direc...
49997    I am a Catholic taught in parochial elementary...
49998    Im going to have to disagree with the previous...
49999    No one expects the Star Trek movies to be high...
Name: review, Length: 50000, dtype: object

In [89]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)


In [90]:
from sklearn.metrics import accuracy_score, classification_report

# training predictions
y_train_pred = model.predict(X_train_tfidf)

# testing predictions
y_test_pred = model.predict(X_test_tfidf)

# accuracies
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

print("\nClassification Report (Test Data):")
print(classification_report(y_test, y_test_pred))

Training Accuracy: 0.92575
Testing Accuracy: 0.9025

Classification Report (Test Data):
              precision    recall  f1-score   support

    negative       0.91      0.90      0.90      5000
    positive       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [95]:
import pandas as pd

sample_text = """
<p>OMG this movie was sooo good!!! 😂😂</p>
I can't------------ believe it lol. []/:::
Check this........... review here: https://www.imdb.com/title/tt0111161/
BRB watching it again. IMO one of the best movies ever!)()
"""

df_1 = pd.DataFrame({
    "review": [sample_text]
})

df_1 = preprocess(df_1)

X_sample = tfidf.transform(df_1["review"])

prediction = model.predict(X_sample)

print("Predicted Sentiment:", prediction[0])

Started with Removing HTML Tags....
completed with Removing HTML Tags....
Started with Remove URL ....
Completed with Remove URL ....
Started with Chat Conversion ....
Completed with Chat Conversion ....
Started with Emoji conversion ....
Completed with Emoji conversion ....
Started with Cleaning Text ....
Completed with Cleaning Text ....
Predicted Sentiment: positive


In [96]:
import os
import joblib

model_dir = "C:\\Users\\kousi\\Documents\\sentiment_analysis_fastapi_aws_docker\\models"

os.makedirs(model_dir, exist_ok=True)

joblib.dump(model, os.path.join(model_dir, "sentiment_model.pkl"))
joblib.dump(tfidf, os.path.join(model_dir, "tfidf_vectorizer.pkl"))

['C:\\Users\\kousi\\Documents\\sentiment_analysis_fastapi_aws_docker\\models\\tfidf_vectorizer.pkl']